# testing learn-to-look

Direction and hour questions. **Select s**, click the board, **Select v**, click, **Select v_bar**, click, then **Generate**. Each click lands on the unit square. The three points stay fixed for the whole reply.

**Load** takes a coordinate checkpoint: a directory with the LoRA (`adapter_config.json`, `adapter_model.safetensors`) and `coord_embed.pt`. The first load builds 4-bit Gemma and applies that adapter. A later load swaps the adapter and the coordinate code. **Refresh** rescans `weights/`.

The question is prefilled with no coordinate code, the same way the trainer did it. Each reply token is then fed back with the same `(s, v, v_bar)`. Decode is greedy. It stops on the end-of-turn token, or at 32 new tokens.

After a click the selection advances from s to v to v_bar. Press a button again to move that point. The readout is the oracle for the three clicks (up/down, left/right, hour): the geometry of the points.

The board still has exactly one gold. The notepad is `target: the gold`, with no region description. The user message is that notepad, the picture, and the question. The question box starts as the trainer's direction question. **Hour** replaces it with the trainer's hour question. You can edit the box after that.

The picture in the prompt is the clean frame. The dots on the widget are the clicks: blue `s`, red `v`, orange `v_bar`.

Run this on the GPU box. The cell needs the repo environment (`pygame`, `torch`, `ipywidgets`).


In [ ]:
from __future__ import annotations

import html
import io
import math
import os
import sys
import tempfile
import time
import traceback
from pathlib import Path

_here = Path.cwd().resolve()
REPO_ROOT = _here.parent if _here.name == "notebooks" else _here
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
import ipywidgets as widgets
from IPython.display import HTML, display
from PIL import Image, ImageDraw

from agent.config import CONFIG
from agent.game_io import (
    apply_edited_settings_dict,
    game_to_settings_dict,
    new_multi_gold_game,
)
from agent.memory import format_notepad
from agent.model import ADAPTERS, spec_for
from agent.modes import SYSTEM_PROMPT_S2, _build_game_messages
from agent.notebook_ui import settings_editor
from neural_net.coord_embed import COORD_INPUT_HIGH, COORD_INPUT_LOW, CoordEmbedder
from neural_net.paths import weights_root
from neural_net.render import canonical_frame
from neural_net.s2 import _hidden_size, _wants_cache_position
from neural_net.s2_coord_oracle import (
    AXIS_MARGIN,
    HOUR_BOUNDARY_DEG,
    KIND_HOUR,
    KIND_LEFTRIGHT,
    KIND_UPDOWN,
    clock_hour,
    correct_line,
    direction_label,
    gaze_vbar,
    hour_is_unique,
)
from neural_net.s2_coord_train import QUESTION_DIRECTION, QUESTION_HOUR
from training.train import TrainConfig, build_model, load_adapter_state, resolve_terminator_id

SPEC_KEY = "gemma-4-12b"
MAX_NEW_TOKENS = 32
DEVICE = "cuda:0"
_FRAME_PATH = Path(tempfile.gettempdir()) / "testing_learn_to_look_frame.png"
_NOTEPAD = format_notepad(
    [{"key": "target", "value": "the gold", "updated_round": 1}]
)
_PRESETS = {
    "direction": QUESTION_DIRECTION,
    "hour": QUESTION_HOUR,
}
_COLORS = {
    "s": (37, 99, 235),
    "v": (220, 38, 38),
    "v_bar": (234, 88, 12),
}
_PICK_NEXT = {"s": "v", "v": "v_bar"}

state = {
    "game": None,
    "frame": None,
    "model": None,
    "processor": None,
    "tokenizer": None,
    "terminator": None,
    "embedder": None,
    "spec": None,
    "ckpt": None,
    "busy": False,
    "s": None,
    "v": None,
    "v_bar": None,
    "ignore_through": 0,
}


def _checkpoints():
    root = weights_root()
    found = []
    if not root.is_dir():
        return found
    for path in root.rglob("coord_embed.pt"):
        parent = path.parent
        if (parent / "adapter_config.json").is_file() and (
            parent / "adapter_model.safetensors"
        ).is_file():
            found.append(parent)
    found.sort(key=lambda item: item.stat().st_mtime, reverse=True)
    return found


def _label(path):
    root = weights_root()
    try:
        return str(path.relative_to(root))
    except ValueError:
        return str(path)


def _options(paths):
    if not paths:
        return [("(no coord checkpoint)", None)]
    return [(_label(path), str(path)) for path in paths]


def _say(text):
    log.append_stdout(text if text.endswith("\n") else text + "\n")


def _set_busy(flag):
    state["busy"] = flag
    for widget in controls:
        widget.disabled = flag
    settings.set_disabled(flag)


def _points():
    missing = [name for name in ("s", "v", "v_bar") if state[name] is None]
    if missing:
        raise RuntimeError(
            "click the board to set " + ", ".join(missing)
        )
    sx_, sy_ = state["s"]
    vx_, vy_ = state["v"]
    vbx_, vby_ = state["v_bar"]
    vals = [float(sx_), float(sy_), float(vx_), float(vy_), float(vbx_), float(vby_)]
    if not all(math.isfinite(v) for v in vals):
        raise RuntimeError("s, v, and v_bar must be finite numbers")
    return vals


def _require_range(vals):
    sx_, sy_, vx_, vy_, vbx_, vby_ = vals
    for name, x, y, lo, hi in (
        ("s", sx_, sy_, 0.0, 1.0),
        ("v", vx_, vy_, 0.0, 1.0),
        ("v_bar", vbx_, vby_, COORD_INPUT_LOW, COORD_INPUT_HIGH),
    ):
        if not (lo <= x <= hi and lo <= y <= hi):
            raise RuntimeError(
                f"{name} ({x}, {y}) is outside [{lo}, {hi}]"
            )


def _on_board(x, y):
    return 0.0 <= x <= 1.0 and 0.0 <= y <= 1.0


def _near(a, b):
    return abs(a[0] - b[0]) < 1e-4 and abs(a[1] - b[1]) < 1e-4


def _oracle_lines(vals):
    sx_, sy_, vx_, vy_, vbx_, vby_ = vals
    lines = []
    up = direction_label(KIND_UPDOWN, sx_, sy_, vx_, vy_)
    side = direction_label(KIND_LEFTRIGHT, sx_, sy_, vx_, vy_)
    if up is None:
        lines.append(f"up/down: |v_y - s_y| < {AXIS_MARGIN}")
    else:
        lines.append(correct_line(KIND_UPDOWN, up))
    if side is None:
        lines.append(f"left/right: |v_x - s_x| < {AXIS_MARGIN}")
    else:
        lines.append(correct_line(KIND_LEFTRIGHT, side))
    hour = clock_hour(sx_, sy_, vx_, vy_)
    hour_line = correct_line(KIND_HOUR, hour)
    if not hour_is_unique(sx_, sy_, vx_, vy_):
        hour_line += f" (within {HOUR_BOUNDARY_DEG:g}° of an hour boundary)"
    lines.append(hour_line)
    reflected = gaze_vbar(sx_, sy_, vx_, vy_, looking=True)
    same = gaze_vbar(sx_, sy_, vx_, vy_, looking=False)
    if _near((vbx_, vby_), reflected):
        lines.append("v_bar = 2s − v")
    elif _near((vbx_, vby_), same):
        lines.append("v_bar = v")
    else:
        lines.append("v_bar is a hand-set point")
    return lines


def _refresh_oracle():
    try:
        vals = _points()
        _require_range(vals)
        lines = _oracle_lines(vals)
    except RuntimeError as exc:
        oracle.value = (
            "<div style='font-family:monospace;color:#a33;font-size:13px'>"
            f"{html.escape(str(exc))}</div>"
        )
        return
    body = "<br>".join(html.escape(line) for line in lines)
    oracle.value = (
        "<div style='font-family:monospace;font-size:13px;color:#222'>"
        f"{body}</div>"
    )


def _one_gold(game):
    golds = game_to_settings_dict(game).get("gold") or []
    if len(golds) != 1:
        raise RuntimeError(
            f"this notebook needs exactly one gold, the board has {len(golds)}"
        )
    return (float(golds[0][0]), float(golds[0][1]))


def _write_frame(frame):
    Image.fromarray(np.ascontiguousarray(frame), mode="RGB").save(_FRAME_PATH)
    return str(_FRAME_PATH)


def _dot(draw, size, x, y, color):
    if not _on_board(x, y):
        return
    width, height = size
    px = x * (width - 1)
    py = (1.0 - y) * (height - 1)
    r = 8
    draw.ellipse((px - r, py - r, px + r, py + r), fill=color, outline=(0, 0, 0))


def _show_board():
    frame = state["frame"]
    if frame is None:
        return
    image = Image.fromarray(np.ascontiguousarray(frame), mode="RGB")
    draw = ImageDraw.Draw(image)
    for name, color in _COLORS.items():
        point = state[name]
        if point is not None:
            _dot(draw, image.size, point[0], point[1], color)
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    board.value = buf.getvalue()


def _fresh_frame():
    state["frame"] = canonical_frame(state["game"])
    _write_frame(state["frame"])
    _show_board()


def _start_game():
    state["game"] = new_multi_gold_game(n_gold=1, opening="any")
    _fresh_frame()


def _messages(question):
    return _build_game_messages(
        SYSTEM_PROMPT_S2,
        str(_FRAME_PATH),
        "",
        question,
        notepad=_NOTEPAD,
    )


def _log_prompt(messages):
    for message in messages:
        role = message.get("role", "?")
        content = message.get("content")
        chunks = []
        if isinstance(content, str):
            chunks.append(content)
        elif isinstance(content, list):
            for part in content:
                if not isinstance(part, dict):
                    continue
                if part.get("type") == "image":
                    chunks.append("[image: current frame]")
                elif part.get("type") == "text":
                    chunks.append(part.get("text") or "")
        _say(f"--- {role} ---\n" + "\n".join(chunks))


def _encode(messages):
    spec = state["spec"]
    norm = ADAPTERS[spec.family].prepare_messages(messages)
    prompt = state["processor"].apply_chat_template(
        norm,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {}
    for key, val in prompt.items():
        if not isinstance(val, torch.Tensor):
            continue
        if val.dtype.is_floating_point:
            val = val.to(torch.bfloat16)
        inputs[key] = val.to(DEVICE)
    if "attention_mask" not in inputs:
        inputs["attention_mask"] = torch.ones_like(inputs["input_ids"])
    return inputs


def _generate_ids(inputs, coord):
    model = state["model"]
    embedder = state["embedder"]
    model.eval()
    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()
    model.config.use_cache = True
    produced = []
    with torch.inference_mode():
        out = model(**inputs, use_cache=True)
        past = out.past_key_values
        logits = out.logits[:, -1, :]
        seq_len = int(inputs["input_ids"].shape[1])
        for _step in range(MAX_NEW_TOKENS):
            next_id = int(logits.argmax(dim=-1).reshape(-1)[0].item())
            if next_id == state["terminator"]:
                break
            produced.append(next_id)
            seq_len += 1
            step = {
                "input_ids": torch.tensor([[next_id]], device=DEVICE),
                "attention_mask": torch.ones(
                    1, seq_len, dtype=torch.long, device=DEVICE
                ),
                "past_key_values": past,
            }
            if _wants_cache_position(model):
                step["cache_position"] = torch.tensor(
                    [seq_len - 1], device=DEVICE
                )
            # Same six floats on every reply token. The prompt, including
            # the question, was prefilled with no code.
            embedder.set_pending(
                coord,
                torch.zeros(1, 1, dtype=torch.long, device=DEVICE),
            )
            try:
                out = model(**step, use_cache=True)
            finally:
                embedder.clear_pending()
            past = out.past_key_values
            logits = out.logits[:, -1, :]
    return produced


def _prepare_model():
    if state["model"] is not None:
        return
    if not torch.cuda.is_available():
        raise RuntimeError(f"this notebook needs {DEVICE}")
    spec = spec_for(SPEC_KEY)
    cfg = TrainConfig(
        label="learn_to_look_test",
        device=DEVICE,
        neftune_alpha=0.0,
    )
    model, processor, _, _ = build_model(spec, cfg, CONFIG.hf_token)
    tokenizer = getattr(processor, "tokenizer", processor)
    state["model"] = model
    state["processor"] = processor
    state["tokenizer"] = tokenizer
    state["terminator"] = resolve_terminator_id(model, tokenizer)
    state["spec"] = spec


def _install_embedder(ckpt):
    model = state["model"]
    if state["embedder"] is not None:
        state["embedder"].uninstall()
        state["embedder"] = None
    embedder = CoordEmbedder(_hidden_size(model))
    embedder.load(ckpt / "coord_embed.pt")
    if not embedder.enabled:
        raise RuntimeError(f"{ckpt / 'coord_embed.pt'} has enabled=False")
    embedder.to(device=DEVICE, dtype=torch.float32)
    embedder.freeze_readout()
    embedder.install(model)
    state["embedder"] = embedder


def on_refresh(_):
    if state["busy"]:
        return
    paths = _checkpoints()
    opts = _options(paths)
    ckpt_dd.options = opts
    ckpt_dd.value = opts[0][1]
    status.value = f"{len(paths)} coord checkpoint(s)."


def on_load(_):
    if state["busy"]:
        return
    chosen = ckpt_dd.value
    if not chosen:
        status.value = "No coord checkpoint. Refresh after the weights are in place."
        return
    ckpt = Path(chosen)
    _set_busy(True)
    status.value = f"Loading { _label(ckpt) }."
    try:
        _prepare_model()
        load_adapter_state(state["model"], ckpt)
        _install_embedder(ckpt)
        state["model"].eval()
        if hasattr(state["model"], "gradient_checkpointing_disable"):
            state["model"].gradient_checkpointing_disable()
        state["model"].config.use_cache = True
        state["ckpt"] = ckpt
        _say(f"loaded {_label(ckpt)}")
        _arm_status()
    except Exception:
        status.value = "Load failed."
        _say(traceback.format_exc())
        raise
    finally:
        _set_busy(False)


def on_preset(change):
    question.value = _PRESETS[change["new"]]


def _arm_text():
    which = pick.value
    if state[which] is None:
        return f"Click the board to set {which}."
    return f"Click the board to move {which}."


def _arm_status():
    status.value = _arm_text()


def _world(x_frac, y_from_top):
    x = min(1.0, max(0.0, float(x_frac)))
    y = min(1.0, max(0.0, 1.0 - float(y_from_top)))
    return x, y


def on_pick(_change):
    if state["busy"]:
        return
    _arm_status()


def on_board_click(change):
    raw = change["new"]
    if not raw or raw.count(",") != 2:
        return
    seq_s, xs, ys = raw.split(",")
    try:
        seq = int(seq_s)
        x_frac = float(xs)
        y_top = float(ys)
    except ValueError:
        return
    if seq <= state["ignore_through"] or state["busy"]:
        return
    state["ignore_through"] = seq
    if settings.is_editing():
        status.value = "Render the settings before clicking the board."
        return
    x, y = _world(x_frac, y_top)
    which = pick.value
    state[which] = (x, y)
    nxt = _PICK_NEXT.get(which)
    if nxt is not None:
        pick.value = nxt
    else:
        _arm_status()
    _refresh_oracle()
    _show_board()


def on_clear(_):
    if state["busy"]:
        return
    log.clear_output()
    _arm_status()


def on_new(_):
    if state["busy"]:
        return
    try:
        _start_game()
    except Exception:
        status.value = "New board failed."
        _say(traceback.format_exc())
        raise
    view = game_to_settings_dict(state["game"])
    settings.show_view(view)
    gold = _one_gold(state["game"])
    _say(
        f"new board  agent=({float(view['agent_x']):.4f}, "
        f"{float(view['agent_y']):.4f})  "
        f"gold=({gold[0]:.4f}, {gold[1]:.4f})  "
        f"openings={len(view.get('openings') or [])}"
    )
    if state["model"] is None:
        status.value = "Load a coord checkpoint. " + _arm_text()
    else:
        _arm_status()


def on_settings_edit(_):
    if state["busy"]:
        return
    settings.enter_edit(game_to_settings_dict(state["game"]))


def on_settings_render(_):
    if state["busy"] or not settings.is_editing():
        return
    try:
        drafted = settings.collect()
        if len(drafted.get("gold") or []) != 1:
            settings.set_error("This notebook needs exactly one gold.")
            return
        state["game"] = apply_edited_settings_dict(drafted)
    except ValueError as exc:
        settings.set_error(f"Error, fix settings before rendering.\n{exc}")
        return
    _fresh_frame()
    settings.show_view(game_to_settings_dict(state["game"]))
    _say("rendered settings")


def on_generate(_):
    if state["busy"]:
        return
    if state["model"] is None or state["embedder"] is None:
        status.value = "Load a coord checkpoint first."
        return
    if settings.is_editing():
        status.value = "Render the settings before Generate."
        return
    question_text = question.value.strip()
    if not question_text:
        status.value = "The question is empty."
        return
    try:
        vals = _points()
        _require_range(vals)
    except RuntimeError as exc:
        status.value = str(exc)
        return
    _set_busy(True)
    status.value = "Generating."
    try:
        if not state["embedder"].enabled:
            raise RuntimeError("the coordinate code is disabled")
        _one_gold(state["game"])
        frame = canonical_frame(state["game"])
        state["frame"] = frame
        _write_frame(frame)
        _show_board()
        sx_, sy_, vx_, vy_, vbx_, vby_ = vals
        _say(
            f"s=({sx_:.4f}, {sy_:.4f})  "
            f"v=({vx_:.4f}, {vy_:.4f})  "
            f"v_bar=({vbx_:.4f}, {vby_:.4f})"
        )
        for line in _oracle_lines(vals):
            _say(line)
        messages = _messages(question_text)
        _log_prompt(messages)
        inputs = _encode(messages)
        prompt_len = int(inputs["input_ids"].shape[1])
        _say(f"prompt tokens {prompt_len} (no coordinate code on the question)")
        _say(f"checkpoint {_label(state['ckpt'])}")
        coord = torch.tensor(
            vals, dtype=torch.float32, device=DEVICE
        ).view(1, 1, 6)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        started = time.perf_counter()
        produced = _generate_ids(inputs, coord)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        text = state["tokenizer"].decode(produced, skip_special_tokens=True)
        why = (
            "end of turn"
            if len(produced) < MAX_NEW_TOKENS
            else f"cap {MAX_NEW_TOKENS}"
        )
        carried = (
            "each reply token carries this s, v, v_bar"
            if produced
            else "the model ended on the end-of-turn token"
        )
        _say(f"{len(produced)} reply tokens in {elapsed:.1f}s ({why}); {carried}")
        _say("reply:")
        _say(text if text else "(empty)")
        _say("")
        _arm_status()
    except Exception:
        status.value = "Generate failed."
        _say(traceback.format_exc())
        raise
    finally:
        _set_busy(False)


_paths = _checkpoints()
_opts = _options(_paths)

ckpt_dd = widgets.Dropdown(
    options=_opts,
    value=_opts[0][1],
    description="Checkpoint:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="520px"),
)
refresh_btn = widgets.Button(description="Refresh", icon="refresh")
load_btn = widgets.Button(description="Load", button_style="primary", icon="download")
refresh_btn.on_click(on_refresh)
load_btn.on_click(on_load)

pick = widgets.ToggleButtons(
    options=[("Select s", "s"), ("Select v", "v"), ("Select v_bar", "v_bar")],
    value="s",
)
pick.observe(on_pick, names="value")

click_box = widgets.Text(value="")
click_box.add_class("ltl-click")
click_box.layout.height = "0px"
click_box.layout.visibility = "hidden"
click_box.layout.overflow = "hidden"
click_box.observe(on_board_click, names="value")

oracle = widgets.HTML()
legend = widgets.HTML(
    "<div style='font-family:sans-serif;font-size:13px;color:#444'>"
    "Select s, click the board, then v, then v_bar. "
    "Blue is s, red is v, orange is v_bar. "
    "The prompt image has no dots.</div>"
)

preset = widgets.Dropdown(
    options=[("Direction", "direction"), ("Hour", "hour")],
    value="direction",
    description="Preset:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="240px"),
)
preset.observe(on_preset, names="value")
question = widgets.Textarea(
    value=QUESTION_DIRECTION,
    layout=widgets.Layout(width="640px", height="140px"),
)

gen_btn = widgets.Button(description="Generate", button_style="success", icon="play")
new_btn = widgets.Button(description="New board", icon="plus")
clear_btn = widgets.Button(description="Clear", icon="trash")
gen_btn.on_click(on_generate)
new_btn.on_click(on_new)
clear_btn.on_click(on_clear)

status = widgets.HTML(
    "<span style='font-family:sans-serif;font-size:13px'>"
    "Load a coord checkpoint.</span>"
)
log = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ccc",
        height="320px",
        overflow="auto",
        width="100%",
    )
)

board = widgets.Image(format="png", width=420)
board.layout = widgets.Layout(width="420px", flex="0 0 420px")
board.add_class("ltl-board")
settings = settings_editor(height_px=360)
settings.edit_btn.on_click(on_settings_edit)
settings.render_btn.on_click(on_settings_render)

controls = [
    ckpt_dd, refresh_btn, load_btn,
    pick, preset, question,
    gen_btn, new_btn, clear_btn,
]

_start_game()
settings.show_view(game_to_settings_dict(state["game"]))
_refresh_oracle()
_arm_status()

display(HTML(
    """
<style>
.ltl-board img { cursor: crosshair; }
</style>
<script>
(function () {
  if (window.__ltlBoardClick) { return; }
  window.__ltlBoardClick = true;
  window.addEventListener("click", function (ev) {
    var img = ev.target;
    if (!img || img.tagName !== "IMG" || !img.closest(".ltl-board")) { return; }
    var r = img.getBoundingClientRect();
    if (!(r.width > 0) || !(r.height > 0)) { return; }
    var x = (ev.clientX - r.left) / r.width;
    var y = (ev.clientY - r.top) / r.height;
    x = Math.min(1, Math.max(0, x));
    y = Math.min(1, Math.max(0, y));
    window.__ltlClickN = (window.__ltlClickN || 0) + 1;
    var box = document.querySelector(".ltl-click input");
    if (!box) { return; }
    box.value = window.__ltlClickN + "," + x + "," + y;
    box.dispatchEvent(new Event("input", { bubbles: true }));
  }, true);
})();
</script>
"""
))
display(widgets.VBox([
    widgets.HBox([
        widgets.VBox([board, legend]),
        settings.box,
    ]),
    widgets.HBox([ckpt_dd, refresh_btn, load_btn]),
    pick,
    oracle,
    widgets.HBox([preset]),
    question,
    widgets.HBox([gen_btn, new_btn, clear_btn]),
    status,
    log,
    click_box,
]))
